In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    AutoConfig,
    PreTrainedModel
)

from torch import nn
from torch.utils.data import DataLoader

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA: 12.1


device(type='cuda')

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
df = pd.read_csv("../data/processed/mhs_comment_soft_labels.csv")

print("Full soft-label shape:", df.shape)
print(df["split"].value_counts())

Full soft-label shape: (19761, 23)
split
train         13832
test           2965
validation     2964
Name: count, dtype: int64


In [5]:
df = pd.read_csv("../data/processed/mhs_comment_soft_labels.csv")

print("Full soft-label shape:", df.shape)
print(df["split"].value_counts())

Full soft-label shape: (19761, 23)
split
train         13832
test           2965
validation     2964
Name: count, dtype: int64


In [6]:
df = df[df["is_women_targeted"] == 1].copy()

print("Women-targeted shape:", df.shape)
print(df["split"].value_counts())

Women-targeted shape: (11983, 23)
split
train         8387
test          1799
validation    1797
Name: count, dtype: int64


In [7]:
ann = pd.read_csv("../data/processed/mhs_main_experiment_annotations_with_split.csv")

In [8]:
ann = ann[ann["is_women_targeted"] == 1].copy()

In [9]:
def make_distribution(group, label_col="hatespeech", label_values=[0, 1, 2]):
    counts = group[label_col].value_counts(normalize=True)
    return pd.Series({
        f"{label_col}_{label}_prob": counts.get(label, 0.0)
        for label in label_values
    })

In [10]:
global_dist = (
    ann.groupby("comment_id")
    .apply(lambda g: make_distribution(g))
    .reset_index()
)

In [11]:
women_dist = (
    ann[ann["annotator_gender_group"] == "women"]
    .groupby("comment_id")
    .apply(lambda g: make_distribution(g))
    .reset_index()
    .rename(columns={
        "hatespeech_0_prob": "women_hatespeech_0_prob",
        "hatespeech_1_prob": "women_hatespeech_1_prob",
        "hatespeech_2_prob": "women_hatespeech_2_prob",
    })
)

In [12]:
men_dist = (
    ann[ann["annotator_gender_group"] == "men"]
    .groupby("comment_id")
    .apply(lambda g: make_distribution(g))
    .reset_index()
    .rename(columns={
        "hatespeech_0_prob": "men_hatespeech_0_prob",
        "hatespeech_1_prob": "men_hatespeech_1_prob",
        "hatespeech_2_prob": "men_hatespeech_2_prob",
    })
)

In [13]:
metadata = (
    ann.groupby("comment_id")
    .agg(
        text_clean=("text_clean", "first"),
        split=("split", "first"),
        target_type=("target_type", "first"),
        annotation_count=("annotator_id", "count")
    )
    .reset_index()
)

In [14]:
df = (
    metadata
    .merge(global_dist, on="comment_id", how="left")
    .merge(women_dist, on="comment_id", how="left")
    .merge(men_dist, on="comment_id", how="left")
)

In [15]:
women_cols = [
    "women_hatespeech_0_prob",
    "women_hatespeech_1_prob",
    "women_hatespeech_2_prob"
]

men_cols = [
    "men_hatespeech_0_prob",
    "men_hatespeech_1_prob",
    "men_hatespeech_2_prob"
]

df["women_mask"] = df[women_cols].notna().all(axis=1).astype(int)
df["men_mask"] = df[men_cols].notna().all(axis=1).astype(int)

df[women_cols + men_cols] = df[women_cols + men_cols].fillna(0.0)

In [16]:
text_col = "text_clean"

global_cols = [
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob"
]

women_cols = [
    "women_hatespeech_0_prob",
    "women_hatespeech_1_prob",
    "women_hatespeech_2_prob"
]

men_cols = [
    "men_hatespeech_0_prob",
    "men_hatespeech_1_prob",
    "men_hatespeech_2_prob"
]

needed_cols = [
    "comment_id",
    text_col,
    "split"
] + global_cols + women_cols + men_cols

df = df[needed_cols].copy()

df.head()

,comment_id,text_clean,split,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob,women_hatespeech_0_prob,women_hatespeech_1_prob,women_hatespeech_2_prob,men_hatespeech_0_prob,men_hatespeech_1_prob,men_hatespeech_2_prob
0,6,First off you look cool as fuck! Anyway if we ...,train,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,11,"eat my fuck, bitch",validation,0.0,0.5,0.5,0.0,1.0,0.0,0.0,0.0,1.0
2,12,She ugly anyways,train,0.5,0.0,0.5,0.5,0.0,0.5,0.0,0.0,0.0
3,19,Won't happen. Birth rates are down. Women want...,train,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,22,The guys are one thing but then you have a wom...,train,0.5,0.0,0.5,0.0,0.0,0.0,0.5,0.0,0.5


In [17]:
df["women_mask"] = (df[women_cols].sum(axis=1) > 0).astype(int)
df["men_mask"] = (df[men_cols].sum(axis=1) > 0).astype(int)

print("Women perspective available:", df["women_mask"].sum())
print("Men perspective available:", df["men_mask"].sum())

df[["women_mask", "men_mask"]].value_counts()

Women perspective available: 8630
Men perspective available: 7168


women_mask  men_mask
1           0           4761
            1           3869
0           1           3299
            0             54
Name: count, dtype: int64

In [18]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (8387, 14)
Validation: (1797, 14)
Test: (1799, 14)


In [19]:
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print("\n", name.upper())

    print("Global label sums:")
    print(part[global_cols].sum(axis=1).describe())

    print("Women label sums where available:")
    print(part.loc[part["women_mask"] == 1, women_cols].sum(axis=1).describe())

    print("Men label sums where available:")
    print(part.loc[part["men_mask"] == 1, men_cols].sum(axis=1).describe())


 TRAIN
Global label sums:
count    8387.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
dtype: float64
Women label sums where available:
count    6021.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
dtype: float64
Men label sums where available:
count    5.020000e+03
mean     1.000000e+00
std      2.216239e-18
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

 VALIDATION
Global label sums:
count    1797.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
dtype: float64
Women label sums where available:
count    1329.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
dtype: float64
Men label sums where available:
count    1061.0
mean        1.0
std         0.

In [20]:
hf_cols = [
    text_col
] + global_cols + women_cols + men_cols + ["women_mask", "men_mask"]

train_dataset = Dataset.from_pandas(train_df[hf_cols].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[hf_cols].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[hf_cols].reset_index(drop=True))

train_dataset

Dataset({
    features: ['text_clean', 'hatespeech_0_prob', 'hatespeech_1_prob', 'hatespeech_2_prob', 'women_hatespeech_0_prob', 'women_hatespeech_1_prob', 'women_hatespeech_2_prob', 'men_hatespeech_0_prob', 'men_hatespeech_1_prob', 'men_hatespeech_2_prob', 'women_mask', 'men_mask'],
    num_rows: 8387
})

In [21]:
model_name = "roberta-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [22]:
def tokenize(batch):
    return tokenizer(
        batch[text_col],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/8387 [00:00<?, ? examples/s]

Map:   0%|          | 0/1797 [00:00<?, ? examples/s]

Map:   0%|          | 0/1799 [00:00<?, ? examples/s]

In [23]:
def add_labels(batch):
    global_labels = []
    women_labels = []
    men_labels = []

    for i in range(len(batch[global_cols[0]])):
        global_labels.append([
            float(batch["hatespeech_0_prob"][i]),
            float(batch["hatespeech_1_prob"][i]),
            float(batch["hatespeech_2_prob"][i])
        ])

        women_labels.append([
            float(batch["women_hatespeech_0_prob"][i]),
            float(batch["women_hatespeech_1_prob"][i]),
            float(batch["women_hatespeech_2_prob"][i])
        ])

        men_labels.append([
            float(batch["men_hatespeech_0_prob"][i]),
            float(batch["men_hatespeech_1_prob"][i]),
            float(batch["men_hatespeech_2_prob"][i])
        ])

    batch["global_labels"] = global_labels
    batch["women_labels"] = women_labels
    batch["men_labels"] = men_labels

    return batch

train_dataset = train_dataset.map(add_labels, batched=True)
val_dataset = val_dataset.map(add_labels, batched=True)
test_dataset = test_dataset.map(add_labels, batched=True)

Map:   0%|          | 0/8387 [00:00<?, ? examples/s]

Map:   0%|          | 0/1797 [00:00<?, ? examples/s]

Map:   0%|          | 0/1799 [00:00<?, ? examples/s]

In [24]:
columns_to_keep = [
    "input_ids",
    "attention_mask",
    "global_labels",
    "women_labels",
    "men_labels",
    "women_mask",
    "men_mask"
]

train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

In [25]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [26]:
class RobertaGenderPerspectiveHeads(PreTrainedModel):
    config_class = AutoConfig

    def __init__(self, config, model_name, num_labels=3, dropout_prob=0.1):
        super().__init__(config)

        self.roberta = AutoModel.from_pretrained(model_name, config=config)

        hidden_size = config.hidden_size

        self.dropout = nn.Dropout(dropout_prob)

        self.global_head = nn.Linear(hidden_size, num_labels)
        self.women_head = nn.Linear(hidden_size, num_labels)
        self.men_head = nn.Linear(hidden_size, num_labels)

        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        global_labels=None,
        women_labels=None,
        men_labels=None,
        women_mask=None,
        men_mask=None,
        **kwargs
    ):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs
        )

        pooled = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)

        global_logits = self.global_head(pooled)
        women_logits = self.women_head(pooled)
        men_logits = self.men_head(pooled)

        return {
            "global_logits": global_logits,
            "women_logits": women_logits,
            "men_logits": men_logits
        }

In [27]:
config = AutoConfig.from_pretrained(model_name)

model = RobertaGenderPerspectiveHeads(
    config=config,
    model_name=model_name,
    num_labels=3,
    dropout_prob=0.1
)

model.to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaGenderPerspectiveHeads(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
            

In [28]:
LAMBDA_GLOBAL = 1.0
LAMBDA_WOMEN = 1.0
LAMBDA_MEN = 1.0

In [29]:
def masked_kl_loss(logits, labels, mask):
    """
    KL loss only for rows where mask == 1.
    """
    mask = mask.float()

    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits.device)

    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    per_sample_loss = torch.sum(
        labels * (torch.log(labels + 1e-12) - log_probs),
        dim=-1
    )

    masked_loss = (per_sample_loss * mask).sum() / mask.sum()

    return masked_loss

In [30]:
class GenderPerspectiveTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        global_labels = inputs.pop("global_labels").float()
        women_labels = inputs.pop("women_labels").float()
        men_labels = inputs.pop("men_labels").float()

        women_mask = inputs.pop("women_mask").float()
        men_mask = inputs.pop("men_mask").float()

        outputs = model(**inputs)

        global_loss = masked_kl_loss(
            outputs["global_logits"],
            global_labels,
            torch.ones_like(women_mask)
        )

        women_loss = masked_kl_loss(
            outputs["women_logits"],
            women_labels,
            women_mask
        )

        men_loss = masked_kl_loss(
            outputs["men_logits"],
            men_labels,
            men_mask
        )

        total_loss = (
            LAMBDA_GLOBAL * global_loss
            + LAMBDA_WOMEN * women_loss
            + LAMBDA_MEN * men_loss
        )

        loss_outputs = {
            "loss": total_loss,
            "global_loss": global_loss.detach(),
            "women_loss": women_loss.detach(),
            "men_loss": men_loss.detach(),
            "global_logits": outputs["global_logits"],
            "women_logits": outputs["women_logits"],
            "men_logits": outputs["men_logits"]
        }

        return (total_loss, loss_outputs) if return_outputs else total_loss

In [31]:
training_args = TrainingArguments(
    output_dir="../models/roberta_gender_perspective_heads_1",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=6,
    weight_decay=0.01,

    logging_steps=100,
    load_best_model_at_end=False,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)

In [32]:
trainer = GenderPerspectiveTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

/home/shayan/Distributional-Hate-Speech-Prediction/.venv/lib/python3.12/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [33]:
torch.cuda.empty_cache()

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.807600,1.788301
2,1.708200,1.800746
3,1.563400,1.913510
4,1.415600,1.996999
5,1.267600,2.175829
6,1.017400,2.236348


TrainOutput(global_step=6294, training_loss=1.448155563446617, metrics={'train_runtime': 605.7884, 'train_samples_per_second': 83.069, 'train_steps_per_second': 10.39, 'total_flos': 1914665396038956.0, 'train_loss': 1.448155563446617, 'epoch': 6.0})

In [34]:
def predict_gender_perspective(model, dataset, batch_size=32):
    model.eval()

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=data_collator
    )

    outputs_all = {
        "global_probs": [],
        "women_probs": [],
        "men_probs": [],
        "global_labels": [],
        "women_labels": [],
        "men_labels": [],
        "women_mask": [],
        "men_mask": []
    }

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        global_labels = batch.pop("global_labels").float()
        women_labels = batch.pop("women_labels").float()
        men_labels = batch.pop("men_labels").float()
        women_mask = batch.pop("women_mask").float()
        men_mask = batch.pop("men_mask").float()

        with torch.no_grad():
            outputs = model(**batch)

        global_probs = torch.nn.functional.softmax(outputs["global_logits"], dim=-1)
        women_probs = torch.nn.functional.softmax(outputs["women_logits"], dim=-1)
        men_probs = torch.nn.functional.softmax(outputs["men_logits"], dim=-1)

        outputs_all["global_probs"].append(global_probs.cpu().numpy())
        outputs_all["women_probs"].append(women_probs.cpu().numpy())
        outputs_all["men_probs"].append(men_probs.cpu().numpy())

        outputs_all["global_labels"].append(global_labels.cpu().numpy())
        outputs_all["women_labels"].append(women_labels.cpu().numpy())
        outputs_all["men_labels"].append(men_labels.cpu().numpy())

        outputs_all["women_mask"].append(women_mask.cpu().numpy())
        outputs_all["men_mask"].append(men_mask.cpu().numpy())

    return {
        key: np.concatenate(value, axis=0)
        for key, value in outputs_all.items()
    }

In [35]:
def distribution_metrics(probs, labels):
    eps = 1e-12

    kl = np.sum(
        labels * (np.log(labels + eps) - np.log(probs + eps)),
        axis=1
    ).mean()

    m = 0.5 * (labels + probs)

    js = 0.5 * (
        np.sum(labels * (np.log(labels + eps) - np.log(m + eps)), axis=1)
        +
        np.sum(probs * (np.log(probs + eps) - np.log(m + eps)), axis=1)
    ).mean()

    pred_hard = probs.argmax(axis=1)
    gold_hard = labels.argmax(axis=1)

    hard_accuracy = (pred_hard == gold_hard).mean()

    pred_entropy = -np.sum(probs * np.log2(probs + eps), axis=1)
    gold_entropy = -np.sum(labels * np.log2(labels + eps), axis=1)

    if np.std(pred_entropy) == 0 or np.std(gold_entropy) == 0:
        entropy_corr = np.nan
    else:
        entropy_corr = np.corrcoef(pred_entropy, gold_entropy)[0, 1]

    return {
        "kl_divergence": float(kl),
        "js_divergence": float(js),
        "hard_accuracy": float(hard_accuracy),
        "entropy_correlation": float(entropy_corr)
    }

In [36]:

def masked_distribution_metrics(probs, labels, mask):
    mask = mask.astype(bool)

    if mask.sum() == 0:
        return {
            "kl_divergence": np.nan,
            "js_divergence": np.nan,
            "hard_accuracy": np.nan,
            "entropy_correlation": np.nan,
            "n_examples": 0
        }

    metrics = distribution_metrics(
        probs[mask],
        labels[mask]
    )

    metrics["n_examples"] = int(mask.sum())

    return metrics

In [37]:
val_pred = predict_gender_perspective(model, val_dataset)
test_pred = predict_gender_perspective(model, test_dataset)

In [38]:
val_results = {
    "global": distribution_metrics(
        val_pred["global_probs"],
        val_pred["global_labels"]
    ),
    "women": masked_distribution_metrics(
        val_pred["women_probs"],
        val_pred["women_labels"],
        val_pred["women_mask"]
    ),
    "men": masked_distribution_metrics(
        val_pred["men_probs"],
        val_pred["men_labels"],
        val_pred["men_mask"]
    )
}

test_results = {
    "global": distribution_metrics(
        test_pred["global_probs"],
        test_pred["global_labels"]
    ),
    "women": masked_distribution_metrics(
        test_pred["women_probs"],
        test_pred["women_labels"],
        test_pred["women_mask"]
    ),
    "men": masked_distribution_metrics(
        test_pred["men_probs"],
        test_pred["men_labels"],
        test_pred["men_mask"]
    )
}

val_results, test_results

({'global': {'kl_divergence': 0.697372317314148,
   'js_divergence': 0.14670558273792267,
   'hard_accuracy': 0.7245409015025042,
   'entropy_correlation': 0.23443575639947112},
  'women': {'kl_divergence': 0.7972725033760071,
   'js_divergence': 0.168034166097641,
   'hard_accuracy': 0.7027840481565086,
   'entropy_correlation': 0.14924175454321933,
   'n_examples': 1329},
  'men': {'kl_divergence': 0.7370324730873108,
   'js_divergence': 0.15793122351169586,
   'hard_accuracy': 0.7276154571159283,
   'entropy_correlation': 0.12521557316797863,
   'n_examples': 1061}},
 {'global': {'kl_divergence': 0.7483980655670166,
   'js_divergence': 0.1552342176437378,
   'hard_accuracy': 0.7015008337965536,
   'entropy_correlation': 0.22863784795775413},
  'women': {'kl_divergence': 0.8269857168197632,
   'js_divergence': 0.17312029004096985,
   'hard_accuracy': 0.70390625,
   'entropy_correlation': 0.11510266581762413,
   'n_examples': 1280},
  'men': {'kl_divergence': 0.8433353900909424,
   'j

In [39]:
model_save_path = "../models/roberta_gender_perspective_heads_1/final_model"

os.makedirs(model_save_path, exist_ok=True)

model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("Saved model to:", model_save_path)

Saved model to: ../models/roberta_gender_perspective_heads_1/final_model


In [40]:
from datetime import datetime
import json

os.makedirs("../results/tables", exist_ok=True)

results_path = "../results/tables/roberta_gender_perspective_heads_1_results.txt"
history_path = "../results/tables/roberta_gender_perspective_heads_1_training_history.csv"

history_df = pd.DataFrame(trainer.state.log_history)
history_df.to_csv(history_path, index=False)

with open(results_path, "w", encoding="utf-8") as f:
    f.write("ROBERTA GENDER PERSPECTIVE HEADS MODEL RESULTS\n")
    f.write("=" * 80 + "\n\n")

    f.write("1. RUN INFORMATION\n")
    f.write("-" * 80 + "\n")
    f.write(f"Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model name: {model_name}\n")
    f.write(f"Output directory: {training_args.output_dir}\n")
    f.write(f"Global training steps: {trainer.state.global_step}\n\n")

    f.write("2. MODEL OBJECTIVE\n")
    f.write("-" * 80 + "\n")
    f.write("Dataset: women-targeted comments only\n")
    f.write("Architecture: shared RoBERTa encoder with three distribution heads\n")
    f.write("- Global annotator distribution head\n")
    f.write("- Women annotator distribution head\n")
    f.write("- Men annotator distribution head\n\n")

    f.write("3. DATASET SIZES\n")
    f.write("-" * 80 + "\n")
    f.write(f"Women-targeted total rows: {len(df)}\n")
    f.write(f"Train rows: {len(train_df)}\n")
    f.write(f"Validation rows: {len(val_df)}\n")
    f.write(f"Test rows: {len(test_df)}\n")
    f.write(f"Women perspective available total: {int(df['women_mask'].sum())}\n")
    f.write(f"Men perspective available total: {int(df['men_mask'].sum())}\n\n")

    f.write("4. TRAINING SETUP\n")
    f.write("-" * 80 + "\n")
    f.write(f"Max sequence length: {MAX_LENGTH}\n")
    f.write(f"Train batch size: {training_args.per_device_train_batch_size}\n")
    f.write(f"Eval batch size: {training_args.per_device_eval_batch_size}\n")
    f.write(f"Epochs: {training_args.num_train_epochs}\n")
    f.write(f"Learning rate: {training_args.learning_rate}\n")
    f.write(f"Weight decay: {training_args.weight_decay}\n")
    f.write(f"FP16: {training_args.fp16}\n\n")

    f.write("5. VALIDATION RESULTS\n")
    f.write("-" * 80 + "\n")
    f.write(json.dumps(val_results, indent=2))
    f.write("\n\n")

    f.write("6. TEST RESULTS\n")
    f.write("-" * 80 + "\n")
    f.write(json.dumps(test_results, indent=2))
    f.write("\n\n")

    f.write("7. TRAINING HISTORY\n")
    f.write("-" * 80 + "\n")
    f.write(f"Training history saved to: {history_path}\n")
    f.write(f"Total log entries: {len(history_df)}\n\n")


print("Saved:", results_path)
print("Saved history:", history_path)

Saved: ../results/tables/roberta_gender_perspective_heads_1_results.txt
Saved history: ../results/tables/roberta_gender_perspective_heads_1_training_history.csv
